# Actividad 1 - Abstracción
**Estudiante:** 20211140003  

### Explicación
La abstracción consiste en tomar solo los datos y métodos que importan para un problema puntual, dejando de lado lo que no hace falta.

Aquí usamos de ejemplo un **Cuarto Frío** en dos situaciones distintas:
1. `CuartoFrioAlmacen`: para el sistema de logística. Solo maneja capacidad de pallets, ocupación y cobro por día. No le importan temas de motores ni presiones.
2. `CuartoFrioMonitoreo`: para el técnico de mantenimiento. Maneja temperatura del sensor, presión del gas y horas de uso del compresor. No le importan los clientes ni los precios.

Ambas clases representan el mismo objeto físico pero abstraído para dos necesidades diferentes.

### Diagrama UML
```plantuml
@startuml
skinparam classAttributeIconSize 0

class CuartoFrioAlmacen {
    + id_cuarto: str
    + capacidad_pallets: int
    + pallets_ocupados: int
    + tarifa_diaria_pallet: float
    --
    + __init__(id_cuarto: str, capacidad_pallets: int, tarifa_diaria_pallet: float)
    + ingresar_carga(pallets: int): bool
    + retirar_carga(pallets: int): bool
    + calcular_facturacion_diaria(): float
    + obtener_porcentaje_ocupacion(): float
    + __str__(): str
}

class CuartoFrioMonitoreo {
    + id_cuarto: str
    + temperatura_sensor_c: float
    + presion_succion_psi: float
    + horas_acumuladas_compresor: float
    + temperatura_critica_c: float
    --
    + __init__(id_cuarto: str, temp_critica: float, horas_iniciales: float)
    + registrar_telemetria(temp_c: float, presion_psi: float, horas_operacion: float): None
    + evaluar_estado_termico(): str
    + requiere_mantenimiento_preventivo(): bool
    + __str__(): str
}
@enduml
```

In [ ]:
# Contexto 1: bodega y logística de carga
class CuartoFrioAlmacen:
    def __init__(self, id_cuarto: str, capacidad_pallets: int, tarifa_diaria_pallet: float) -> None:
        self.id_cuarto: str = id_cuarto
        self.capacidad_pallets: int = capacidad_pallets
        self.pallets_ocupados: int = 0
        self.tarifa_diaria_pallet: float = tarifa_diaria_pallet

    def ingresar_carga(self, pallets: int) -> bool:
        # revisar que quepan en el cuarto
        if pallets > 0 and self.pallets_ocupados + pallets <= self.capacidad_pallets:
            self.pallets_ocupados += pallets
            print(f"[{self.id_cuarto}] Ingresaron {pallets} pallets. Ocupados: {self.pallets_ocupados}/{self.capacidad_pallets}")
            return True
        print(f"[{self.id_cuarto}] No caben {pallets} pallets. Disponibles: {self.capacidad_pallets - self.pallets_ocupados}")
        return False

    def retirar_carga(self, pallets: int) -> bool:
        if 0 < pallets <= self.pallets_ocupados:
            self.pallets_ocupados -= pallets
            print(f"[{self.id_cuarto}] Salieron {pallets} pallets. Quedan: {self.pallets_ocupados}")
            return True
        print(f"[{self.id_cuarto}] No se pueden sacar {pallets} pallets.")
        return False

    def calcular_facturacion_diaria(self) -> float:
        return self.pallets_ocupados * self.tarifa_diaria_pallet

    def obtener_porcentaje_ocupacion(self) -> float:
        if self.capacidad_pallets > 0:
            return (self.pallets_ocupados / self.capacidad_pallets) * 100.0
        return 0.0

    def __str__(self) -> str:
        return f"Cuarto Logístico {self.id_cuarto} | {self.pallets_ocupados}/{self.capacidad_pallets} pallets ({self.obtener_porcentaje_ocupacion():.1f}%)"


# Contexto 2: monitoreo técnico y mantenimiento
class CuartoFrioMonitoreo:
    def __init__(self, id_cuarto: str, temp_critica: float, horas_iniciales: float = 0.0) -> None:
        self.id_cuarto: str = id_cuarto
        self.temperatura_sensor_c: float = 0.0
        self.presion_succion_psi: float = 0.0
        self.horas_acumuladas_compresor: float = horas_iniciales
        self.temperatura_critica_c: float = temp_critica

    def registrar_telemetria(self, temp_c: float, presion_psi: float, horas_operacion: float) -> None:
        self.temperatura_sensor_c = temp_c
        self.presion_succion_psi = presion_psi
        if horas_operacion > 0:
            self.horas_acumuladas_compresor += horas_operacion

    def evaluar_estado_termico(self) -> str:
        if self.temperatura_sensor_c > self.temperatura_critica_c:
            return f"Alerta: temperatura {self.temperatura_sensor_c}°C pasó el límite ({self.temperatura_critica_c}°C)"
        return f"Normal: temperatura {self.temperatura_sensor_c}°C"

    def requiere_mantenimiento_preventivo(self) -> bool:
        # se revisa cada 5000 horas de uso
        return self.horas_acumuladas_compresor >= 5000.0

    def __str__(self) -> str:
        return f"Cuarto Técnico {self.id_cuarto} | Temp: {self.temperatura_sensor_c}°C | Presión: {self.presion_succion_psi} PSI | Horas: {self.horas_acumuladas_compresor:.1f}h"

print("Clases de abstracción definidas correctamente.")


In [ ]:
# Probamos el cuarto en modo logistica
print("--- Probando Cuarto Logística ---")
c_almacen = CuartoFrioAlmacen("CF-01", 100, 15.0)
print(c_almacen)
c_almacen.ingresar_carga(60)
c_almacen.ingresar_carga(50)  # no cabe
c_almacen.retirar_carga(10)
print(f"Cobro del día: ${c_almacen.calcular_facturacion_diaria():.2f}")
print(c_almacen)

# Probamos el cuarto en modo mantenimiento
print("\n--- Probando Cuarto Mantenimiento ---")
c_tecnico = CuartoFrioMonitoreo("CF-01", temp_critica=4.0, horas_iniciales=4980.0)
print(c_tecnico)
c_tecnico.registrar_telemetria(temp_c=2.5, presion_psi=28.0, horas_operacion=15.0)
print(c_tecnico.evaluar_estado_termico())
print(f"¿Toca mantenimiento?: {c_tecnico.requiere_mantenimiento_preventivo()}")

c_tecnico.registrar_telemetria(temp_c=5.8, presion_psi=34.0, horas_operacion=10.0)
print(c_tecnico.evaluar_estado_termico())
print(f"¿Toca mantenimiento?: {c_tecnico.requiere_mantenimiento_preventivo()}")
print(c_tecnico)
